In [ ]:
import os
import shutil
from pyspark.sql import SparkSession
# ... other imports ...

# --- Configuration ---
ICEBERG_VERSION = "1.5.0"
LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
CATALOG_NAME = "bronze"

# --- Stop existing SparkSession ---
try:
    if 'spark' in globals() and spark is not None:
        spark.stop()
except Exception:
    pass

# # --- Clean warehouse ---
# # Note: For production use, should not be cleaned the warehouse every time!
# if os.path.exists(LOCAL_WAREHOUSE_PATH):
#     print(f"Cleaning up old warehouse: {LOCAL_WAREHOUSE_PATH}")
#     # Using os.path.join for robust path handling on Windows
#     # The f"file://{LOCAL_WAREHOUSE_PATH}" is for the Spark config, not os.path.exists
#     try:
#         shutil.rmtree(LOCAL_WAREHOUSE_PATH)
#     except Exception as e:
#         print(f"Warning: Could not fully clean path. Please ensure no files are open. Error: {e}")


# --- Iceberg packages ---
ICEBERG_PACKAGES = (
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION},"
    f"org.apache.avro:avro:1.11.3"
)

## --- SparkSession ---
spark = SparkSession.builder \
    .appName("IcebergDescribeExample") \
    .config("spark.jars.packages", ICEBERG_PACKAGES) \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "hadoop") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.executor.memory", "12g") \
    .config("spark.executor.cores", "4") \
    .config("spark.driver.maxResultSize", "12g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.shuffle.compress", "true") \
    .config("spark.shuffle.spill.compress", "true") \
    \
    .getOrCreate()

print("Spark version:", spark.version)
print(f"Spark Session and Iceberg Warehouse is set to: {spark.conf.get(f'spark.sql.catalog.{CATALOG_NAME}.warehouse')}")

In [ ]:
from pyspark.sql import SparkSession

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
CATALOG_NAME = "bronze"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

print("Spark session started with Iceberg support.")

In [ ]:
SalesOrderHeader = spark.table("bronze.manual_Sales.SalesOrderHeader")
SalesOrderHeader.count()

In [ ]:
df_newSOH = spark.sql("""SELECT
    SalesOrderID
,   CAST(RevisionNumber AS   int) RevisionNumber
,   CAST(OrderDate AS    date) OrderDate
,   CAST(DueDate AS  date) DueDate
,   CAST(ShipDate AS     date) ShipDate
,   CAST(Status AS   int) Status
,   CAST(OnlineOrderFlag AS  boolean) OnlineOrderFlag
,   SalesOrderNumber
,   PurchaseOrderNumber
,   AccountNumber
,   CAST(CustomerID AS   int) CustomerID
,   CAST(SalesPersonID AS    int) SalesPersonID
,   CAST(TerritoryID AS  int) TerritoryID
,   CAST(BillToAddressID AS  int) BillToAddressID
,   CAST(ShipToAddressID AS  int) ShipToAddressID
,   CAST(ShipMethodID AS     int) ShipMethodID
,   CAST(CreditCardID AS     int) CreditCardID
,   CreditCardApprovalCode
,   CAST(CurrencyRateID AS   int) CurrencyRateID
,   CAST(SubTotal AS     Decimal(12, 4)) SubTotal
,   CAST(TaxAmt AS   Decimal(12, 4)) TaxAmt
,   CAST(Freight AS  Decimal(12, 4)) Freight
,   CAST(TotalDue AS     Decimal(12, 4)) TotalDue
,   rowguid
,   comment
FROM bronze.Sales.SalesOrderHeader
          WHERE SalesOrderID NOT IN (Select SalesOrderID FROM bronze.manual_Sales.SalesOrderHeader)
""")
df_newSOH.show(2)

In [ ]:
df_newSOH.writeTo("bronze.manual_Sales.SalesOrderHeader").append()

In [ ]:
spark.sql("""
INSERT INTO bronze.manual_Sales.SalesOrderHeader
(
    SalesOrderID
,	RevisionNumber
,	OrderDate
,	DueDate
,	ShipDate
,	Status
,	OnlineOrderFlag
,	SalesOrderNumber
,	PurchaseOrderNumber
,	AccountNumber
,	CustomerID
,	SalesPersonID
,	TerritoryID
,	BillToAddressID
,	ShipToAddressID
,	ShipMethodID
,	CreditCardID
,	CreditCardApprovalCode
,	CurrencyRateID
,	SubTotal
,	TaxAmt
,	Freight
,	TotalDue
,	rowguid
,   comment
,	ModifiedDate
)
SELECT
    SalesOrderID
,   CAST(RevisionNumber AS   int) RevisionNumber
,   CAST(OrderDate AS    date) OrderDate
,   CAST(DueDate AS  date) DueDate
,   CAST(ShipDate AS     date) ShipDate
,   CAST(Status AS   int) Status
,   CAST(OnlineOrderFlag AS  boolean) OnlineOrderFlag
,   SalesOrderNumber
,   PurchaseOrderNumber
,   AccountNumber
,   CAST(CustomerID AS   int) CustomerID
,   CAST(SalesPersonID AS    int) SalesPersonID
,   CAST(TerritoryID AS  int) TerritoryID
,   CAST(BillToAddressID AS  int) BillToAddressID
,   CAST(ShipToAddressID AS  int) ShipToAddressID
,   CAST(ShipMethodID AS     int) ShipMethodID
,   CAST(CreditCardID AS     int) CreditCardID
,   CreditCardApprovalCode
,   CAST(CurrencyRateID AS   int) CurrencyRateID
,   CAST(SubTotal AS     Decimal(12, 4)) SubTotal
,   CAST(TaxAmt AS   Decimal(12, 4)) TaxAmt
,   CAST(Freight AS  Decimal(12, 4)) Freight
,   CAST(TotalDue AS     Decimal(12, 4)) TotalDue
,   rowguid
,   comment
,   CAST(ModifiedDate AS     date) ModifiedDate
FROM bronze.Sales.SalesOrderHeader
          WHERE SalesOrderID NOT IN (Select SalesOrderID FROM bronze.manual_Sales.SalesOrderHeader)
          LIMIT 5000
""")


In [ ]:
# soh = spark.table("bronze.Sales.SalesOrderHeader").alias("soh").limit(1000)
soh = spark.table("bronze.Sales.SalesOrderHeader").alias("soh").filter("year(to_date(ModifiedDate, 'M/d/yyyy')) = '2017'").limit(1000)
# soh.printSchema()
soh.show()


In [ ]:
# Drop old tables first (if needed)
spark.sql("DROP TABLE IF EXISTS bronze.manual_Person.BusinessEntityContact")
spark.sql("DROP TABLE IF EXISTS bronze.manual_Person.ContactType")

# Then create fresh ones
spark.sql("""
CREATE TABLE bronze.manual_Person.BusinessEntityContact(
    BusinessEntityID INT,
    PersonID INT,
    ContactTypeID INT,
    rowguid STRING,
    ModifiedDate DATE
) USING iceberg
""")

spark.sql("""
CREATE TABLE bronze.manual_Person.ContactType(
    ContactTypeID INT,
    Name STRING,
    ModifiedDate DATE
) USING iceberg
""")

In [ ]:
spark.sql("""INSERT INTO bronze.manual_Person.ContactType 
          (ContactTypeID, Name, ModifiedDate) 
          SELECT CAST(ContactTypeID as INT), Name, 
                 CAST(ModifiedDate as DATE) 
          FROM bronze.Person.ContactType""")


In [ ]:
spark.sql("""INSERT INTO 
          bronze.manual_Person.BusinessEntityContact 
          (BusinessEntityID, PersonID, ContactTypeID, rowguid, ModifiedDate) 
          SELECT 
          CAST(BusinessEntityID AS INT), 
          CAST(PersonID AS INT), 
          CAST(ContactTypeID AS INT), 
          rowguid, 
          CAST(ModifiedDate as DATE) 
          FROM bronze.Person.BusinessEntityContact""")

In [ ]:
from pyspark.sql import functions as F

dfsql = spark.sql("""Select BEC.BusinessEntityID, BEC.PersonID, BEC.ContactTypeID, BEC.rowguid, BEC.ModifiedDate, CT.ContactTypeID, CT.Name, CT.ModifiedDate 
          FROM bronze.Person.ContactType CT INNER JOIN
          bronze.Person.BusinessEntityContact BEC
          ON CT.ContactTypeID = BEC.ContactTypeID
          """)
dfsql.show(10)

In [ ]:
from pyspark.sql import functions as F

dfsql = spark.sql("""Select BEC.BusinessEntityID, BEC.PersonID, BEC.ContactTypeID, BEC.rowguid, BEC.ModifiedDate, CT.ContactTypeID, CT.Name, CT.ModifiedDate 
          FROM bronze.manual.ContactType CT INNER JOIN
          bronze.manual.BusinessEntityContact BEC
          ON CT.ContactTypeID = BEC.ContactTypeID
          """)
dfsql.show(10)

In [ ]:
df_CT = spark.table("local.manualcreated.ContactType").alias("CT")
df_BEC = spark.table("local.manualcreated.BusinessEntityContact").alias("BEC")

df_joined = df_CT.join(df_BEC, on="ContactTypeID", how="inner")
df_joined.select("BEC.PersonID", F.col("CT.Name").alias("ContactType")).show(50)

In [ ]:
from pyspark.sql import functions as F

df_CT = spark.table("local.manual.ContactType").alias("CT")
df_BEC = spark.table("local.manual.BusinessEntityContact").alias("BEC")

# df_CT.show(2)
# df_BEC.show(2)

df_output = df_CT.join(
    df_BEC, F.col("CT.ContactTypeID") == F.col("BEC.ContactTypeID"), "inner"
)


spark.sql("""
CREATE TABLE IF NOT EXISTS local.manual.PersonContactType (
    PersonID INT,
    ContactType STRING,
    insertDate DATE
)
USING iceberg
PARTITIONED BY (year(insertDate))
""")

df_load = df_output.select("BEC.PersonID", F.col("CT.Name").alias("ContactType"), F.col("BEC.ModifiedDate").alias("insertDate"))

df_load.writeTo("local.manual.PersonContactType").append() #.overwritePartitions()



In [ ]:
# Load Data using saved CSV files.

from pyspark.sql import functions as F
import random

LOCAL_ICEBERG_CATALOG = 'local'
folder_path = "C:/data/data_files/tmp/" # Replace with your actual folder path
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
dataframes = {}

for file_name in csv_files:
    print(file_name + ' - ' + file_name[:file_name.find("_")] + ' - '  + file_name[file_name.find("_") + 1:-4])
    file_path = os.path.join(folder_path, file_name)
    print("filepath: " + file_path)
    file_name_noext = os.path.splitext(file_name)[0] # Get filename without extension
    print(file_name_noext)

    # # Read the CSV file into a DataFrame
    # # Adjust options like header and inferSchema as needed
    df_csv = spark.read.csv(file_path, header=True, inferSchema=True)

    # This will be the database name (e.g., 'mydb')
    db_name = file_name[:file_name.find("_")] 
    
    # This will be the table name (e.g., 'mytable')
    table_name = file_name[file_name.find("_") + 1:-4]
    
    # 2. Construct the Fully Qualified Iceberg Table Identifier
    # Iceberg requires a three-part name: catalog.database.table
    fq_iceberg_table = f"{LOCAL_ICEBERG_CATALOG}.{db_name}.{table_name}"
    

    print(f"Creating Iceberg table: {fq_iceberg_table}")

        # 3. Create the Iceberg table and write data using the DataFrameWriter API

    df_csv \
        .write \
        .format("iceberg") \
        .mode("overwrite") \
        .saveAsTable(fq_iceberg_table)
    
    # 4. Read the data back from the new Iceberg table
    iceberg_df = spark.read.table(fq_iceberg_table)
    iceberg_df.count()

    


In [ ]:
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
import os

LOCAL_ICEBERG_CATALOG = 'bronze'
folder_path = "C:/data/data_files/tmp/"
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

for file_name in csv_files:
    print(file_name + ' - ' + file_name[:file_name.find("_")] + ' - '  + file_name[file_name.find("_") + 1:-4])
    file_path = os.path.join(folder_path, file_name)
    print("filepath: " + file_path)
    file_name_noext = os.path.splitext(file_name)[0]

    # Read CSV with header, no schema inference
    df_raw = spark.read.csv(file_path, header=True, inferSchema=False)

    # Cast all columns to StringType
    df_csv = df_raw.select([df_raw[col].cast(StringType()).alias(col) for col in df_raw.columns])

    # Extract database and table name
    db_name = file_name[:file_name.find("_")]
    table_name = file_name[file_name.find("_") + 1:-4]
    fq_iceberg_table = f"{LOCAL_ICEBERG_CATALOG}.{db_name}.{table_name}"

    print(f"Creating Iceberg table: {fq_iceberg_table}")

    # Create Iceberg table with all columns as STRING
    schema_sql = ",\n    ".join([f"{col} STRING" for col in df_csv.columns])
    create_sql = f"""
    CREATE TABLE IF NOT EXISTS {fq_iceberg_table} (
        {schema_sql}
    )
    USING iceberg
    """
    spark.sql(create_sql)

    # Write data to Iceberg table
    df_csv.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(fq_iceberg_table)

    # Optional: read back and validate
    iceberg_df = spark.read.table(fq_iceberg_table)
    print(f"{fq_iceberg_table} row count:", iceberg_df.count())

In [ ]:
file_path = "C:/data/data_files/tmp/Sales_SalesOrderHeader.csv"
df_csv = spark.read.csv(file_path, header=True, inferSchema=False)
df_csv.show(10, False)

In [ ]:
spark.sql(" DROP TABLE IF exists bronze.manual_Sales.SalesOrderHeader")

In [ ]:
spark.sql(""" Create Table IF NOT exists bronze.manual_Sales.SalesOrderHeader
          (
          SalesOrderID    string
          ,	RevisionNumber  int
          ,	OrderDate   date
          ,	DueDate date
          ,	ShipDate    date
          ,	Status  int
          ,	OnlineOrderFlag boolean
          ,	SalesOrderNumber    string
          ,	PurchaseOrderNumber string
          ,	AccountNumber   string
          ,	CustomerID  int
          ,	SalesPersonID   int
          ,	TerritoryID int
          ,	BillToAddressID int
          ,	ShipToAddressID int
          ,	ShipMethodID    int
          ,	CreditCardID    int
          ,	CreditCardApprovalCode  string
          ,	CurrencyRateID  int
          ,	SubTotal    Decimal(12, 4)
          ,	TaxAmt  Decimal(12, 4)
          ,	Freight Decimal(12, 4)
          ,	TotalDue    Decimal(12, 4)
          ,	Comment string
          ,	rowguid string
          ,	ModifiedDate    date
          )
""")

In [ ]:
dfTest = spark.sql("""
SELECT
    SalesOrderID
,   CAST(RevisionNumber AS   int) RevisionNumber
,   CAST(OrderDate AS    date) OrderDate
,   CAST(DueDate AS  date) DueDate
,   CAST(ShipDate AS     date) ShipDate
,   CAST(Status AS   int) Status
,   CAST(OnlineOrderFlag AS  boolean) OnlineOrderFlag
,   SalesOrderNumber
,   PurchaseOrderNumber
,   AccountNumber
,   CAST(CustomerID AS   int) CustomerID
,   CAST(SalesPersonID AS    int) SalesPersonID
,   CAST(TerritoryID AS  int) TerritoryID
,   CAST(BillToAddressID AS  int) BillToAddressID
,   CAST(ShipToAddressID AS  int) ShipToAddressID
,   CAST(ShipMethodID AS     int) ShipMethodID
,   CAST(CreditCardID AS     int) CreditCardID
,   CreditCardApprovalCode
,   CAST(CurrencyRateID AS   int) CurrencyRateID
,   CAST(SubTotal AS     Decimal(12, 4)) SubTotal
,   CAST(TaxAmt AS   Decimal(12, 4)) TaxAmt
,   CAST(Freight AS  Decimal(12, 4)) Freight
,   CAST(TotalDue AS     Decimal(12, 4)) TotalDue
,   rowguid
,   CAST(ModifiedDate AS     date) ModifiedDate
FROM bronze.Sales.SalesOrderHeader
""")
dfTest.show(10)

In [ ]:
from pyspark.sql import functions as F

soh = spark.table("bronze.Sales.SalesOrderHeader").alias("soh").limit(5000)
c = spark.table("bronze.Sales.Customer").alias("c").limit(5000)
p = spark.table("bronze.Person.Person").alias("p").limit(5000)
sod = spark.table("bronze.Sales.SalesOrderDetail").alias("sod").limit(5000)
prd = spark.table("bronze.Production.Product").alias("prd").limit(5000)

# soh = dataframes.get("Sales_SalesOrderHeader").alias("soh")
# c = dataframes.get("Sales_Customer").alias("c")
# p = dataframes.get("Person_Person").alias("p")
# sod = dataframes.get("Sales_SalesOrderDetail").alias("sod")
# prd = dataframes.get("Production_Product").alias("prd")
# prd.printSchema()
# sod.printSchema()
# soh.printSchema()
# c.printSchema()

joined_df = soh.join(
    c,
    soh["CustomerID"] == c["CustomerID"],
    how="inner"
) \
.join(
    p, c["PersonID"] == p["BusinessEntityID"],
    how="inner"
) \
.join(
    sod, soh["SalesOrderID"] == sod["SalesOrderID"],
    how="inner"
) \
.join(
    prd, sod["ProductID"] == prd["ProductID"],
    how="inner"
) \
.select(
    F.col("soh.SalesOrderID"),
    F.col("soh.OrderDate"),
    F.col("soh.DueDate"),
    F.col("soh.ShipDate"),
    F.col("soh.Status"),
    F.col("soh.OnlineOrderFlag"),
    F.col("soh.SalesOrderNumber"),
    F.col("soh.PurchaseOrderNumber"),
    F.col("soh.SubTotal"),
    F.col("soh.TaxAmt"),
    F.col("soh.Freight"),
    F.col("soh.TotalDue"),
    F.col("soh.Comment"),
    F.col("c.CustomerID"),
    F.col("p.firstName"),
    F.col("p.lastName"),
    F.col("c.AccountNumber"),
    F.col("prd.name").alias("ProductName"),
    F.col("sod.OrderQty"),
    F.col("sod.UnitPrice"),
    F.col("sod.LineTotal"),
    F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy"))
)

joined_df.show(10)

# joined_df.show(5, truncate=False)
# mayFilter = joined_df.filter((F.year(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 2011) & (F.month(F.to_date(F.col("OrderDate"),"M/d/yyyy")) == 5) & (F.col("AccountNumber") == "AW00029825"))

# mayFilter.show(5, truncate=False)


In [ ]:
spark.stop()